# Suite DIA — The Numpy, Matlab and Excel dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, and an evaluator with the language's own rules: numpy's vectorized, promoting, masked
arithmetic; MATLAB's two-valued logic over doubles, logicals and strings; Excel's coercion and error values. (The
Basic dialect is suites EXP and EVL; what every dialect shares is suite FRM.)

In [ ]:
import math

import numpy as np

from mbse.Expressions.Dialects.Excel import Domains as XD, Evaluators as XV, Expressions as X
from mbse.Expressions.Dialects.Matlab import Domains as MD, Evaluators as MV, Expressions as M
from mbse.Expressions.Dialects.Numpy import Domains as ND, Evaluators as NV, Expressions as N
from mbse.Expressions.Framework import Domains as D
from mbse.Schemas.Framework import Proxies, Schemas as S
from support import raises, text

Person = S.OfObject.Builder().properties(text("name"), text("age", int), text("email")).create()
Proxies.register("Person", Person)
ann = Proxies.Builders.Person().name("Ann").age(30).email("ann@example.com").create()
bob = Proxies.Builders.Person().name("Bob").age(70).create()  # no email

## DIA-01 · Numpy: function-call form, rendered as numpy code

In [ ]:
t = N.name("t")
adult = N.call("greater_equal", N.subscript(t, "age"), 18)
emailed = N.call("logical_not", N.call("ma.getmaskarray", N.subscript(t, "email")))
rule = N.call("logical_and", adult, emailed)
assert N.render(rule) == ("np.logical_and(np.greater_equal(t['age'], 18), "
                          "np.logical_not(np.ma.getmaskarray(t['email'])))")
assert [N.render(N.constant(v)) for v in [math.nan, math.inf, -math.inf, 1.5, "s", b"\x00", True]] == [
    "np.nan", "np.inf", "-np.inf", "1.5", "'s'", "b'\\x00'", "True"]
assert rule.validate(bound={"t"}, core=True) == [] and N.call("isin", 1, 2).validate(core=True) == [
    "'isin' is not a core operation"]
assert N.DIALECT.infer(rule, {"t": ND.Record}) is ND.Bool
assert N.DIALECT.infer(N.call("add", 1, 1.5)) is ND.Float and N.DIALECT.infer(N.call("add", 1, 1)) is ND.Int
assert N.DIALECT.infer(N.call("add", True, True)) is ND.Bool and N.DIALECT.infer(N.call("equal", "a", "b")) is ND.Bool
with raises(TypeError, match="subtract cannot take (bool, bool)"):
    N.DIALECT.infer(N.call("subtract", True, True))  # as numpy refuses
assert ND.Bool.contains(True) and ND.Int.contains(np.array([1, 2])) and not ND.Float.contains(1)
assert ND.Str.contains(np.array(["a"])) and ND.Bytes.contains(b"x") and repr(ND.Number) == "int | float"
assert ND.Numeric.includes(ND.Int) and not ND.Int.includes(ND.Numeric) and ND.OfDtype("any", "biuf").includes(ND.Number)
assert ND.Record.contains({}) and ND.Record.contains(ann) and not ND.Record.contains(1) and repr(ND.Float) == "float"

## DIA-02 · Numpy evaluates vectorized, promotes types and masks missing values

In [ ]:
columns = {"age": np.array([30, 10, 70]), "email": np.ma.array(["a", "b", "c"], mask=[False, False, True])}
assert NV.OfAny(rule, {"t": columns}).tolist() == [True, False, False]
assert NV.OfAny(rule, {"t": ann}) and not NV.OfAny(rule, {"t": bob})  # an absent property is masked
assert NV.OfAny(N.subscript(t, "email"), {"t": bob}) is np.ma.masked
assert NV.OfAny(N.call("add", 1, 1.5)) == 2.5 and NV.OfAny(N.call("equal", 1, 1.0))  # numpy promotes
assert NV.OfAny(N.call("negative", N.subscript(t, "age")), {"t": columns}).tolist() == [-30, -10, -70]
assert NV.OfAny(N.call("where", adult, "adult", "minor"), {"t": columns}).tolist() == ["adult", "minor", "adult"]
records = np.array([(1, 2.0)], dtype=[("a", int), ("b", float)])  # a structured array is a record
assert NV.OfAny(N.subscript(t, "b"), {"t": records}).tolist() == [2.0]
masked = NV.OfAny(N.call("greater", N.subscript(t, "email"), "a"), {"t": columns})
assert masked.mask.tolist() == [False, False, True]  # masks propagate
with raises(NameError, match="name 'u' is not defined"):
    NV.OfAny(N.name("u"))
with raises(KeyError, match="height"):
    NV.OfAny(N.subscript(t, "height"), {"t": columns})
with raises(TypeError, match="cannot subscript a int with 'age'"):
    NV.OfAny(N.subscript(t, "age"), {"t": 3})

## DIA-03 · Matlab: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
s = M.identifier("s")
rule = M.binary("&&", M.binary(">=", M.field(s, "age"), 18), M.call("isfield", s, "email"))
assert M.render(rule) == 's.age >= 18 && isfield(s, "email")'
assert M.render(M.binary("-", 1, M.binary("-", 2, 3))) == "1 - (2 - 3)" and M.render(
    M.binary("-", M.binary("-", 1, 2), 3)) == "1 - 2 - 3"
assert M.render(M.binary(".*", M.binary("+", 1, 2), M.unary("-", M.unary("~", s)))) == "(1 + 2) .* -~s"
assert M.render(M.unary("-", M.binary("+", 1, 2))) == "-(1 + 2)" and M.render(M.unary("~", -1)) == "~-1"
assert M.render(M.binary("||", M.binary("&&", True, False), M.binary("&&", 1, 0))) == "true && false || 1 && 0"
assert [M.render(M.constant(v)) for v in [math.nan, math.inf, -math.inf, -2.5, 'say "hi"', 7]] == [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]
assert M.render(M.field(M.field(s, "a"), "b")) == "s.a.b"
assert M.DIALECT.infer(rule, {"s": MD.Struct}) is MD.Logical
assert M.DIALECT.infer(M.binary("+", "a", 1)) is MD.String and M.DIALECT.infer(M.binary("+", True, 1)) is MD.Double
assert MD.Double.contains(1) and MD.Struct.contains({}) and not MD.Struct.contains("s") and MD.of(1) is MD.Double
assert M.binary("^", 1, 2).validate(core=True) == ["'^' is not a core operation"]

## DIA-04 · Matlab evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
assert MV.OfAny(rule, {"s": ann}) is True and MV.OfAny(rule, {"s": bob}) is False
assert MV.OfAny(rule, {"s": {"age": 20, "email": "x"}}) is True  # a mapping is a struct
assert MV.OfAny(M.field(s, "age"), {"s": ann}) == 30.0 and type(MV.OfAny(18)) is float  # numbers are doubles
assert MV.OfAny(M.binary("==", 1, True)) is True and MV.OfAny(M.binary("+", True, 1)) == 2.0
assert MV.OfAny(M.binary("<", "apple", "banana")) is True and MV.OfAny(M.binary("==", "5", 5)) is True
assert MV.OfAny(M.binary("==", "true", True)) is True and MV.OfAny(M.binary("~=", "1.5", 1.5)) is False
assert MV.OfAny(M.binary("+", "a", 1)) == "a1" and MV.OfAny(M.binary("+", "x", 0.125)) == "x0.125"
assert [MV.OfAny(M.binary(op, 2, 3)) for op in ["~=", "<=", ">", ">=", "-", ".*"]] == [True, True, False, False,
                                                                                        -1.0, 6.0]
assert MV.OfAny(M.binary("&&", False, M.identifier("missing"))) is False  # short-circuits
assert MV.OfAny(M.binary("||", 2, M.identifier("missing"))) is True and MV.OfAny(M.binary("||", 0, 0)) is False
assert MV.OfAny(M.unary("~", 0)) is True and MV.OfAny(M.unary("-", True)) == -1.0
assert MV.OfAny(M.call("isfield", 3, "x")) is False and MV.OfAny(M.call("isfield", s, 1), {"s": ann}) is False
with raises(KeyError, match='Unrecognized field name "email".'):
    MV.OfAny(M.field(s, "email"), {"s": bob})
with raises(TypeError, match="Dot indexing is not supported for variables of this type."):
    MV.OfAny(M.field(s, "age"), {"s": 3})
with raises(NameError, match="Unrecognized function or variable 'missing'."):
    MV.OfAny(M.identifier("missing"))
with raises(TypeError, match="Operands to the && operator must be convertible to logical scalar values."):
    MV.OfAny(M.binary("&&", "yes", True))
with raises(ValueError, match="NaN's cannot be converted to logicals."):
    MV.OfAny(M.unary("~", math.nan))
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string' and 'double'."):
    MV.OfAny(M.binary("-", "a", 1))
with raises(TypeError, match="Operator '.*' is not supported for operands of type 'struct' and 'logical'."):
    MV.OfAny(M.binary(".*", s, True), {"s": ann})
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string'."):
    MV.OfAny(M.unary("-", "a"))

## DIA-05 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
r = X.name("r")
rule = X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18),
                                                     X.function("NOT", X.function("ISERROR", X.field(r, "email")))))
assert X.render(rule) == "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))"
assert X.render(X.infix("*", X.infix("+", 1, 2), X.prefix("-", X.infix("-", 3, 4)))) == "=(1 + 2) * -(3 - 4)"
assert X.render(X.infix("=", X.infix("-", 1, X.infix("-", 2, 3)), -1)) == "=1 - (2 - 3) = -1"
assert X.render(X.prefix("-", X.field(r, "first name"))) == "=-r.[first name]"
assert [X.render(X.constant(v)) for v in [True, False, 'say "hi"', math.inf, 2.5, 3]] == [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]
assert rule.validate(bound={"r"}, core=True) == [] and X.function("VLOOKUP", 1, 2).validate(core=True) == [
    "'VLOOKUP' is not a core operation"]
assert X.DIALECT.infer(rule, {"r": XD.Record}) is XD.Logical and X.DIALECT.infer(X.infix("+", "1", True)) is XD.Number
assert XD.of(1) is XD.Number and XD.Errors.contains(XD.Error("#N/A")) and repr(XD.Error("#N/A")) == "#N/A"
assert XD.Record.contains(ann) and not XD.Record.contains(1)

## DIA-06 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
Error = XV.Error
assert XV.OfAny(rule, {"r": ann}) is True and XV.OfAny(rule, {"r": bob}) is False  # ISERROR finds the #FIELD!
assert XV.OfAny(X.field(r, "email"), {"r": bob}) == Error("#FIELD!") and XV.OfAny(X.name("nope")) == Error("#NAME?")
assert XV.OfAny(X.field(r, "age"), {"r": {"age": 5}}) == 5.0 and XV.OfAny(X.field(r, "x"), {"r": 3}) == Error("#VALUE!")
assert XV.OfAny(X.field(X.field(r, "a"), "b"), {"r": {}}) == Error("#FIELD!")  # errors propagate
assert XV.OfAny(X.infix("+", "1", True)) == 2.0 and XV.OfAny(X.infix("*", "x", 1)) == Error("#VALUE!")  # coercion
assert XV.OfAny(X.infix("-", 1, X.infix("+", X.name("u"), 1))) == Error("#NAME?")
assert XV.OfAny(X.prefix("-", "2")) == -2.0 and XV.OfAny(X.prefix("-", X.name("u"))) == Error("#NAME?")
assert XV.OfAny(X.infix(">", "a", 1)) is True and XV.OfAny(X.infix(">", True, "z")) is True  # number < text < logical
assert XV.OfAny(X.infix("=", "a", "A")) is True and XV.OfAny(X.infix("<>", 1, "1")) is True  # case; no coercion
assert [XV.OfAny(X.infix(op, 2, 3)) for op in ["<", "<=", ">=", "-"]] == [True, True, False, -1.0]
assert XV.OfAny(X.infix("=", X.name("u"), 1)) == Error("#NAME?") and XV.OfAny(X.infix("=", r, 1), {"r": {}}) == Error(
    "#VALUE!")
assert XV.OfAny(X.function("AND", 1, True)) is True and XV.OfAny(X.function("OR", 0, False)) is False
assert XV.OfAny(X.function("AND", False, X.name("u"))) == Error("#NAME?")  # AND evaluates every argument
assert XV.OfAny(X.function("OR", "yes", True)) == Error("#VALUE!") and XV.OfAny(X.function("NOT", 0)) is True
assert XV.OfAny(X.function("NOT", X.name("u"))) == Error("#NAME?")
assert XV.OfAny(X.function("IF", True, 1, X.name("u"))) == 1.0 and XV.OfAny(X.function("IF", 0, 1, "no")) == "no"
assert XV.OfAny(X.function("IF", X.name("u"), 1, 2)) == Error("#NAME?")  # IF evaluates only the branch it takes
assert XV.OfAny(X.function("ISERROR", 1)) is False and XV.OfAny(X.let_("x", 2, X.infix("*", X.name("x"), 3))) == 6.0